# FedAvg Server

> The core abstraction for `fedavg` server.

In [ ]:
#| default_exp servers.fedavg

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os

from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry

<torch._C.Generator>

## ServerFedAvg

In [ ]:
#| export
@AlgorithmRegistry.register_server("fedavg")
class ServerFedAvg(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
                 

We need to make sure that all clients start from the same initial model at round 0. So, we need to save the initial model once at the server side and load it at each client during its initialization.

### Aggregation


The aggegation for `fedavg` is defined as:

$$m_t \leftarrow \sum_{k \in S_t} n_k$$
$$W_{global}^{(t + 1)} \leftarrow \sum_{k \in S_t} \frac{n_k}{m_t} w_k^{(t + 1)}$$

where $S_t$ is the set of active clients at communication round $t$, $n_k$ is the number of samples at client $k$, and $w_k^{(t)}$ is the model parameters of client $k$ after local training at round $t$.

In [ ]:
#| export
@patch
def aggregate(self: ServerFedAvg, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    with torch.no_grad():
        global_model = None
        for id in lst_active_ids:
            client_state_dict = self.state_mgr.get_state(id).get('model', None)
            if global_model is None:
                global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}
            n_k = len_clients_ds[id]
            weight = n_k / m_t
            for key in global_model.keys():
                if not client_state_dict[key].is_floating_point():
                    continue
                global_model[key].add_(client_state_dict[key], alpha=weight)

        # copy integer buffers directly from last client (they should all be the same)
        for key in global_model.keys():
            if not global_model[key].is_floating_point():
                global_model[key].copy_(client_state_dict[key])

        self.model.load_state_dict(global_model)

        for id in lst_active_ids:
            self.state_mgr.set_state(id, {
                'model': self.model.state_dict(),
                'optimizer': self.state_mgr.get_state(id).get('optimizer', None),
            })

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()